# CSCI 467 Project Midterm: Gemstone Recognition

This notebook runs everything the midterm report needs:

1. Downloads the Kaggle *Gemstones Images* dataset and makes a stratified train/dev split.
2. Trains two baselines on hand-made features: logistic regression on a color histogram, and a Random Forest on color plus texture (LBP) features.
3. Trains a small CNN from scratch, as a check on how badly a network overfits with about 30 images per class.
4. Main method: transfer learning from an ImageNet-pretrained ResNet-50, both with only the last layer trained and with the last two blocks fine-tuned.
5. Picks hyperparameters on the dev set, then evaluates each method once on the test set.
6. Saves figures, LaTeX tables, number macros, and the exact train/dev/test split into `results/` for the report and the repo.
7. Saves model weights into `runs/` (too large for GitHub; keep them on Google Drive).

**How to run:** in Colab, choose *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. The full run takes roughly 40 to 60 minutes on a T4. At the end it downloads `results.zip`. In the repo, its contents go in `results/midterm/`, and `results/midterm/splits/` goes in `data/splits/`.

Set `QUICK = True` in the next cell for a short smoke test with tiny settings (numbers are meaningless).

In [ ]:
# ---------- Configuration ----------
QUICK = False            # True = tiny smoke test (fast, numbers meaningless)
SEED = 467
DEV_FRAC = 0.15          # share of the training folder held out as the dev set
IMG = 128                # input size for the small CNN trained from scratch
TL_IMG = 224             # input size for the pretrained ResNet-50 (its native size)
LOAD = 256               # images are stored at this size, then cropped and resized per model
PRETRAINED = True        # use ImageNet weights for ResNet-50 (only the smoke test may fall back)
DATA_DIR = None          # leave None to download from Kaggle, or point at an unzipped copy
OUT_DIR = "results"       # small outputs for the report (committed to the repo)
RUNS_DIR = "runs"         # model weights (not committed)
RUN_TEST = True          # evaluate the chosen models on the test set once at the end

import os, sys, json, time, random, glob, math, shutil
import numpy as np
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
if QUICK:
    TL_IMG = 64

In [ ]:
# ---------- Get the data ----------
# Public Kaggle datasets download without an API key through kagglehub.
# If this fails, download the zip from
# https://www.kaggle.com/datasets/lsind18/gemstones-images
# upload it to Colab, unzip it, and set DATA_DIR to that folder.
if DATA_DIR is None:
    try:
        import kagglehub
    except ImportError:
        !pip -q install kagglehub
        import kagglehub
    DATA_DIR = kagglehub.dataset_download("lsind18/gemstones-images")
print("Data folder:", DATA_DIR)

def find_split_dir(root, name):
    # Find the folder called `name` that holds one subfolder per class.
    hits = [d for d in glob.glob(os.path.join(root, "**", name), recursive=True)
            if os.path.isdir(d) and any(os.path.isdir(os.path.join(d, c)) for c in os.listdir(d))]
    if not hits:
        raise FileNotFoundError(f"No '{name}' folder with class subfolders under {root}")
    return sorted(hits, key=len)[0]

TRAIN_DIR = find_split_dir(DATA_DIR, "train")
TEST_DIR = find_split_dir(DATA_DIR, "test")
print("train:", TRAIN_DIR, "\ntest: ", TEST_DIR)

In [ ]:
# ---------- File lists and the train/dev split ----------
import pandas as pd
from sklearn.model_selection import train_test_split

EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
def list_images(split_dir):
    rows = []
    for cls in sorted(os.listdir(split_dir)):
        d = os.path.join(split_dir, cls)
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(EXTS):
                rows.append((os.path.join(d, f), cls))
    return pd.DataFrame(rows, columns=["path", "label"])

full_train = list_images(TRAIN_DIR)
test_df = list_images(TEST_DIR)
CLASSES = sorted(full_train.label.unique())
C = len(CLASSES)
cls_to_idx = {c: i for i, c in enumerate(CLASSES)}

# Drop test images whose class never appears in training (should be none).
unknown = sorted(set(test_df.label) - set(CLASSES))
if unknown:
    print("Warning: test classes not in train, dropped:", unknown)
    test_df = test_df[test_df.label.isin(CLASSES)].reset_index(drop=True)

train_df, dev_df = train_test_split(full_train, test_size=DEV_FRAC,
                                    stratify=full_train.label, random_state=SEED)
train_df = train_df.reset_index(drop=True); dev_df = dev_df.reset_index(drop=True)
for df in (train_df, dev_df, test_df, full_train):
    df["y"] = df.label.map(cls_to_idx)

counts = pd.DataFrame({
    "train": train_df.label.value_counts(),
    "dev": dev_df.label.value_counts(),
    "test": test_df.label.value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
counts.to_csv(f"{OUT_DIR}/class_counts.csv")

# Save the exact split, with paths relative to the dataset folder, so anyone can reuse it.
DATA_ROOT = os.path.commonpath([TRAIN_DIR, TEST_DIR])
os.makedirs(f"{OUT_DIR}/splits", exist_ok=True)
for name, df in (("train", train_df), ("dev", dev_df), ("test", test_df)):
    out = df[["path", "label"]].copy()
    out["path"] = out.path.map(lambda p: os.path.relpath(p, DATA_ROOT).replace(os.sep, "/"))
    out.to_csv(f"{OUT_DIR}/splits/{name}.csv", index=False)
print(f"{C} classes | train {len(train_df)} | dev {len(dev_df)} | test {len(test_df)}")
print(counts.describe().loc[["min", "50%", "mean", "max"]].round(1))

In [ ]:
# ---------- Class-count figure ----------
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8, "font.family": "serif", "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.bbox": "tight"})

order = counts.sort_values("train", ascending=False)
fig, ax = plt.subplots(figsize=(6.75, 1.9))
x = np.arange(C)
ax.bar(x, order["train"], color="#2a6f97", label="train", width=0.85)
ax.bar(x, order["dev"], bottom=order["train"], color="#e0a458", label="dev", width=0.85)
ax.set_xlim(-1, C)
ax.set_xticks([])
ax.set_xlabel(f"Gemstone class ({C} classes, sorted by training count)")
ax.set_ylabel("Images")
ax.legend(frameon=False, ncol=2, loc="upper right")
fig.savefig(f"{OUT_DIR}/fig_class_counts.pdf"); plt.show()

In [ ]:
# ---------- Load every image once into memory ----------
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor

def load_img(path, size=LOAD):
    im = Image.open(path)
    try:
        im = ImageOps.exif_transpose(im)   # apply phone-camera rotation if recorded
    except Exception:
        pass  # a few files have broken EXIF metadata; use them as stored
    im = im.convert("RGB")
    # Keep the aspect ratio: resize the short side, then center crop to a square.
    return np.asarray(ImageOps.fit(im, (size, size), Image.BILINEAR), dtype=np.uint8)

def load_all(df):
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(load_img, df.path)))

t0 = time.time()
X_train, X_dev, X_test = load_all(train_df), load_all(dev_df), load_all(test_df)
y_train, y_dev, y_test = train_df.y.values, dev_df.y.values, test_df.y.values
print(X_train.shape, X_dev.shape, X_test.shape, f"{time.time()-t0:.0f}s")

In [ ]:
# ---------- Metrics ----------
from sklearn.metrics import f1_score, confusion_matrix

def evaluate(probs, y):
    # probs: (n, C) scores. Returns top-1, top-5, macro F1 (all as fractions).
    pred = probs.argmax(1)
    top5 = np.argsort(-probs, axis=1)[:, :5]
    return {
        "top1": float((pred == y).mean()),
        "top5": float((top5 == y[:, None]).any(1).mean()),
        "macro_f1": float(f1_score(y, pred, labels=np.arange(C), average="macro", zero_division=0)),
    }

results = {}   # method -> {"dev": {...}, "test": {...}, "hparams": {...}}
sweeps = []    # rows of (method, setting, dev top-1, dev macro F1)

## Baselines

**Majority class** always predicts the most common training class.

**Color histogram + logistic regression.** Each image becomes a joint RGB histogram with 8 bins per channel (512 numbers), normalized to sum to 1. A square-root transform evens out the large bins (usually the background), then features are standardized and fed to multinomial logistic regression. The regularization strength `C` is tuned on dev.

**Color + texture + Random Forest.** This follows the best classical setup from Chow and Reyes-Aldasoro (2022): the same RGB histogram plus a hue/saturation histogram and multi-scale uniform local binary patterns (LBP) for texture. The number of trees and minimum leaf size are tuned on dev.

In [ ]:
# ---------- Hand-made features ----------
from skimage.feature import local_binary_pattern
from skimage.color import rgb2hsv, rgb2gray

FEAT_SIZE = 96  # features are computed on a 96x96 copy (faster, same information)

def rgb_hist(img, bins=8):
    q = (img // (256 // bins)).reshape(-1, 3).astype(np.int64)
    idx = q[:, 0] * bins * bins + q[:, 1] * bins + q[:, 2]
    h = np.bincount(idx, minlength=bins ** 3).astype(np.float32)
    return h / h.sum()

def hs_hist(img, hb=12, sb=4):
    hsv = rgb2hsv(img)
    h, _, _ = np.histogram2d(hsv[..., 0].ravel(), hsv[..., 1].ravel(), bins=[hb, sb], range=[[0, 1], [0, 1]])
    h = h.ravel().astype(np.float32)
    return h / h.sum()

def lbp_hist(img):
    g = (rgb2gray(img) * 255).astype(np.uint8)
    feats = []
    for P, R in [(8, 1), (16, 2), (24, 3)]:
        codes = local_binary_pattern(g, P, R, method="uniform")
        h, _ = np.histogram(codes, bins=P + 2, range=(0, P + 2))
        feats.append(h / h.sum())
    return np.concatenate(feats).astype(np.float32)

def features(X):
    small = [np.asarray(Image.fromarray(x).resize((FEAT_SIZE, FEAT_SIZE), Image.BILINEAR)) for x in X]
    color = np.stack([rgb_hist(x) for x in small])
    texture = np.stack([np.concatenate([hs_hist(x), lbp_hist(x)]) for x in small])
    return color, texture

t0 = time.time()
(Fc_tr, Ft_tr), (Fc_dev, Ft_dev), (Fc_te, Ft_te) = features(X_train), features(X_dev), features(X_test)
print("color", Fc_tr.shape, "| texture", Ft_tr.shape, f"| {time.time()-t0:.0f}s")

In [ ]:
# ---------- Baseline 0: majority class ----------
counts_tr = np.bincount(y_train, minlength=C).astype(float)
maj = np.tile(counts_tr, (len(y_dev), 1))
results["Majority class"] = {"dev": evaluate(maj, y_dev), "hparams": {}}
print(results["Majority class"])

In [ ]:
# ---------- Baseline 1: color histogram + logistic regression ----------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def lr_model(c):
    return make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=300 if QUICK else 2000))

Cs = [0.01, 0.1] if QUICK else [0.001, 0.01, 0.1, 1.0]
best = None
for c in Cs:
    m = lr_model(c).fit(np.sqrt(Fc_tr), y_train)
    r = evaluate(m.predict_proba(np.sqrt(Fc_dev)), y_dev)
    sweeps.append(("LogReg (color)", f"C = {c:g}", r["top1"], r["macro_f1"]))
    print(f"C={c:<6g} dev top1={r['top1']:.3f}  macroF1={r['macro_f1']:.3f}")
    if best is None or r["top1"] > best[1]["top1"]:
        best = (c, r, m)
results["LogReg (color)"] = {"dev": best[1], "hparams": {"C": best[0]}, "_model": best[2]}
print("chosen C =", best[0])

In [ ]:
# ---------- Baseline 2: color + texture + Random Forest ----------
from sklearn.ensemble import RandomForestClassifier

F_tr = np.hstack([Fc_tr, Ft_tr]); F_dev = np.hstack([Fc_dev, Ft_dev]); F_te = np.hstack([Fc_te, Ft_te])
grid = [(100, 1), (300, 2)] if QUICK else [(300, 1), (300, 2), (300, 5), (800, 1)]
best = None
for n_trees, leaf in grid:
    m = RandomForestClassifier(n_estimators=n_trees, min_samples_leaf=leaf, max_features="sqrt",
                               n_jobs=-1, random_state=SEED).fit(F_tr, y_train)
    r = evaluate(m.predict_proba(F_dev), y_dev)
    sweeps.append(("RF (color + LBP)", f"{n_trees} trees, min leaf {leaf}", r["top1"], r["macro_f1"]))
    print(f"trees={n_trees:<4} leaf={leaf}  dev top1={r['top1']:.3f}  macroF1={r['macro_f1']:.3f}")
    if best is None or r["top1"] > best[1]["top1"]:
        best = ((n_trees, leaf), r, m)
results["RF (color + LBP)"] = {"dev": best[1], "hparams": {"trees": best[0][0], "min_leaf": best[0][1]},
                               "_model": best[2]}
print("chosen:", best[0])

## A CNN trained from scratch (overfitting check)

The proposal feedback warned that with about 30 images per class, a CNN trained from scratch will likely overfit heavily. This section tests that directly, so the report can compare it against transfer learning below.

Four convolution blocks (two 3x3 convolutions with batch norm and ReLU, then 2x2 max pooling) take a 128x128 image down to an 8x8 map with 256 channels. Global average pooling and dropout feed a linear layer with one output per class. Training uses cross-entropy with light label smoothing and AdamW with a one-cycle learning-rate schedule.

Training augmentation: random resized crops, horizontal and vertical flips, and rotations, since a gem looks the same upside down. Color jitter is tested as a hyperparameter, because color is the strongest signal for telling gems apart and jitter may destroy it.

The epoch with the best dev top-1 accuracy is kept.

In [ ]:
# ---------- CNN ----------
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T

device = "cuda" if torch.cuda.is_available() else "cpu"
# Windows cannot start DataLoader worker processes for classes defined inside a notebook.
NUM_WORKERS = 0 if os.name == "nt" else 2
print("device:", device, "| loader workers:", NUM_WORKERS)
if device == "cpu" and not QUICK:
    print("WARNING: no GPU found. The ResNet-50 runs will take many hours on a CPU. Use Colab with a T4 GPU.")
torch.manual_seed(SEED)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class GemSet(Dataset):
    # Training: random crop, flips, rotation (+ optional color jitter). Eval: resize, then center crop.
    def __init__(self, X, y, train, size=IMG, jitter=False):
        self.X, self.y = torch.from_numpy(X).permute(0, 3, 1, 2), torch.from_numpy(y).long()
        aug = [T.RandomResizedCrop(size, scale=(0.6, 1.0), antialias=True),
               T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(30)]
        if jitter:
            aug.append(T.ColorJitter(0.2, 0.2, 0.2, 0.03))
        self.tf = T.Compose(aug) if train else T.Compose([T.Resize(int(size * 8 / 7), antialias=True), T.CenterCrop(size)])
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        x = self.tf(self.X[i]).float() / 255.0
        return (x - MEAN) / STD, self.y[i]

def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))

class SmallCNN(nn.Module):
    def __init__(self, n_classes, width=32, dropout=0.3):
        super().__init__()
        w = width
        self.features = nn.Sequential(block(3, w), block(w, 2 * w), block(2 * w, 4 * w), block(4 * w, 8 * w))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout),
                                  nn.Linear(8 * w, n_classes))
    def forward(self, x):
        return self.head(self.features(x))

@torch.no_grad()
def predict(model, X, y, size=IMG):
    model.eval()
    dl = DataLoader(GemSet(X, y, train=False, size=size), batch_size=128)
    out = []
    for xb, _ in dl:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            out.append(F.softmax(model(xb.to(device)).float(), 1).cpu())
    return torch.cat(out).numpy()

def train_net(model, param_groups, epochs, size=IMG, jitter=False, wd=5e-4, batch=64, freeze_bn=None, log_every=5):
    # Generic loop: AdamW + one-cycle schedule, keeps the epoch with the best dev top-1.
    # param_groups: list of (parameters, peak learning rate). freeze_bn: modules kept in eval mode.
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = model.to(device)
    opt = torch.optim.AdamW([{"params": p, "lr": lr} for p, lr in param_groups], weight_decay=wd)
    dl = DataLoader(GemSet(X_train, y_train, train=True, size=size, jitter=jitter), batch_size=batch,
                    shuffle=True, num_workers=NUM_WORKERS, drop_last=True, pin_memory=(device == "cuda"))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr for _, lr in param_groups],
                                                total_steps=epochs * len(dl), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    hist, best_state, best_acc = [], None, -1
    for ep in range(1, epochs + 1):
        model.train()
        if freeze_bn is not None:
            freeze_bn.eval()
        tot, correct, n = 0.0, 0, 0
        for xb, yb in dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                logits = model(xb)
                loss = F.cross_entropy(logits.float(), yb, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
        dev = evaluate(predict(model, X_dev, y_dev, size), y_dev)
        hist.append({"epoch": ep, "train_loss": tot / n, "train_acc": correct / n, **{f"dev_{k}": v for k, v in dev.items()}})
        if dev["top1"] > best_acc:
            best_acc = dev["top1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if ep % log_every == 0 or ep == epochs:
            print(f"  ep {ep:3d}  loss {tot/n:.3f}  train acc {correct/n:.3f}  dev top1 {dev['top1']:.3f}")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

def train_cnn(lr, jitter, epochs):
    m = SmallCNN(C)
    return train_net(m, [(m.parameters(), lr)], epochs, size=IMG, jitter=jitter)

print(f"parameters: {sum(p.numel() for p in SmallCNN(C).parameters())/1e6:.2f}M")

In [ ]:
# ---------- CNN hyperparameter sweep on dev ----------
EPOCHS = 3 if QUICK else 60
configs = [(3e-3, False), (1e-3, False)] if QUICK else [(3e-3, False), (1e-3, False), (3e-3, True), (1e-3, True)]
runs = {}
for lr, jitter in configs:
    name = f"lr {lr:g}, color jitter {'on' if jitter else 'off'}"
    print(name)
    t0 = time.time()
    model, hist = train_cnn(lr, jitter, EPOCHS)
    r = evaluate(predict(model, X_dev, y_dev), y_dev)
    best_ep = int(hist.loc[hist.dev_top1.idxmax(), "epoch"])
    runs[name] = (model, hist, r, {"lr": lr, "color_jitter": jitter, "epochs": EPOCHS, "best_epoch": best_ep})
    sweeps.append(("Small CNN", name, r["top1"], r["macro_f1"]))
    print(f"  -> dev top1 {r['top1']:.3f} (best epoch {best_ep}), {time.time()-t0:.0f}s\n")

best_name = max(runs, key=lambda k: runs[k][2]["top1"])
cnn, cnn_hist, cnn_dev, cnn_hp = runs[best_name]
results["Small CNN"] = {"dev": cnn_dev, "hparams": cnn_hp, "_model": cnn}
torch.save(cnn.state_dict(), f"{RUNS_DIR}/cnn_best.pt")
print("chosen:", best_name)

## Main method: transfer learning with a pretrained ResNet-50

ResNet-50 pretrained on ImageNet already knows general visual features (edges, color, texture, shape). We replace its final 1000-way layer with dropout plus a new 87-way layer and train in two ways:

* **Linear probe:** the whole pretrained network is frozen; only the new last layer is trained.
* **Partial fine-tuning:** the last two residual stages (`layer3`, `layer4`) are trained too, with a learning rate 10x smaller than the new layer so the pretrained features change gently. Earlier layers stay frozen.

Images are fed at 224x224, the size ResNet-50 was trained on, with the same flips, crops, and rotations as before. The backbone learning rate is tuned on dev.

In [ ]:
# ---------- ResNet-50 transfer learning ----------
import torchvision

def make_resnet(mode):
    weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if PRETRAINED else None
    try:
        net = torchvision.models.resnet50(weights=weights)
    except Exception as e:
        if not QUICK:
            raise
        print("WARNING: could not download pretrained weights, using random init for the smoke test:", e)
        net = torchvision.models.resnet50(weights=None)
    net.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(net.fc.in_features, C))
    for p in net.parameters():
        p.requires_grad = False
    trainable = [net.fc] if mode == "linear" else [net.layer3, net.layer4, net.fc]
    for m in trainable:
        for p in m.parameters():
            p.requires_grad = True
    return net

class FrozenPart(nn.Module):
    # Wraps the frozen modules so their batch-norm statistics stay fixed during training.
    def __init__(self, net, mode):
        super().__init__()
        keep = ["conv1", "bn1", "layer1", "layer2"] + (["layer3", "layer4"] if mode == "linear" else [])
        self.parts = nn.ModuleList([getattr(net, k) for k in keep])

def train_tl(mode, backbone_lr, head_lr, epochs):
    net = make_resnet(mode)
    groups = [(net.fc.parameters(), head_lr)]
    if mode == "finetune":
        groups.append((list(net.layer3.parameters()) + list(net.layer4.parameters()), backbone_lr))
    return train_net(net, groups, epochs, size=TL_IMG, jitter=False, wd=1e-4, batch=32,
                     freeze_bn=FrozenPart(net, mode))

TL_EPOCHS_LIN = 1 if QUICK else 15
TL_EPOCHS_FT = 1 if QUICK else 25
tl_runs = {}

print("linear probe, head lr 1e-3")
t0 = time.time()
model, hist = train_tl("linear", None, 1e-3, TL_EPOCHS_LIN)
r = evaluate(predict(model, X_dev, y_dev, TL_IMG), y_dev)
sweeps.append(("ResNet-50 (linear probe)", "head lr 0.001", r["top1"], r["macro_f1"]))
results["ResNet-50 (linear probe)"] = {"dev": r, "hparams": {"head_lr": 1e-3, "epochs": TL_EPOCHS_LIN}, "_model": model, "_size": TL_IMG}
lin_hist = hist
print(f"  -> dev top1 {r['top1']:.3f}, {time.time()-t0:.0f}s\n")

for blr in ([1e-4] if QUICK else [1e-4, 3e-4]):
    name = f"backbone lr {blr:g}, head lr 0.001"
    print("fine-tune,", name)
    t0 = time.time()
    model, hist = train_tl("finetune", blr, 1e-3, TL_EPOCHS_FT)
    r = evaluate(predict(model, X_dev, y_dev, TL_IMG), y_dev)
    best_ep = int(hist.loc[hist.dev_top1.idxmax(), "epoch"])
    tl_runs[name] = (model, hist, r, {"backbone_lr": blr, "head_lr": 1e-3, "epochs": TL_EPOCHS_FT, "best_epoch": best_ep})
    sweeps.append(("ResNet-50 (fine-tuned)", name, r["top1"], r["macro_f1"]))
    print(f"  -> dev top1 {r['top1']:.3f} (best epoch {best_ep}), {time.time()-t0:.0f}s\n")

best_name = max(tl_runs, key=lambda k: tl_runs[k][2]["top1"])
ft, ft_hist, ft_dev, ft_hp = tl_runs[best_name]
results["ResNet-50 (fine-tuned)"] = {"dev": ft_dev, "hparams": ft_hp, "_model": ft, "_size": TL_IMG}
results["Small CNN"]["_size"] = IMG
torch.save(ft.state_dict(), f"{RUNS_DIR}/resnet50_ft_best.pt")
print("chosen:", best_name)

In [ ]:
# ---------- Overfitting: scratch CNN vs fine-tuned ResNet-50 ----------
fig, axes = plt.subplots(1, 2, figsize=(3.25, 1.75), sharey=True)
for ax, (title, h) in zip(axes, [("Small CNN (scratch)", cnn_hist), ("ResNet-50 (fine-tuned)", ft_hist)]):
    ax.plot(h.epoch, h.train_acc, color="#2a6f97", label="train")
    ax.plot(h.epoch, h.dev_top1, color="#e0a458", label="dev")
    ax.set_title(title, fontsize=7); ax.set_xlabel("Epoch"); ax.set_ylim(0, 1)
axes[0].set_ylabel("Top-1 accuracy")
axes[1].legend(frameon=False, loc="lower right", fontsize=6)
fig.tight_layout(pad=0.4); fig.savefig(f"{OUT_DIR}/fig_curves.pdf"); plt.show()

def gap(h):
    # train accuracy minus dev accuracy, averaged over the last 5 epochs
    t = h.tail(min(5, len(h)))
    return float((t.train_acc - t.dev_top1).mean())
print(f"train-dev gap  scratch CNN: {gap(cnn_hist):.3f}   fine-tuned ResNet-50: {gap(ft_hist):.3f}")

In [ ]:
# ---------- Final evaluation on the test set (run once) ----------
if RUN_TEST:
    results["Majority class"]["test"] = evaluate(np.tile(counts_tr, (len(y_test), 1)), y_test)
    results["LogReg (color)"]["test"] = evaluate(results["LogReg (color)"]["_model"].predict_proba(np.sqrt(Fc_te)), y_test)
    results["RF (color + LBP)"]["test"] = evaluate(results["RF (color + LBP)"]["_model"].predict_proba(F_te), y_test)
    results["Small CNN"]["test"] = evaluate(predict(cnn, X_test, y_test, IMG), y_test)
    for m in ("ResNet-50 (linear probe)", "ResNet-50 (fine-tuned)"):
        results[m]["test"] = evaluate(predict(results[m]["_model"], X_test, y_test, TL_IMG), y_test)

summary = pd.DataFrame({m: {f"{s} {k}": v.get(s, {}).get(k, np.nan) for s in ("dev", "test")
                            for k in ("top1", "top5", "macro_f1")} for m, v in results.items()}).T
print((summary * 100).round(1))

## Error analysis

The report needs a manual look at a random sample of dev mistakes. The next cell takes the learned model with the best dev accuracy (normally the fine-tuned ResNet-50), samples 24 of its random errors, saves them to `error_sample.csv`, and shows 16 of them in a grid, with the true and predicted labels, plus the most frequently confused class pairs. Look through the grid and note patterns (background, lighting, cut, look-alike colors, label noise).

In [ ]:
# ---------- Error analysis on the dev set ----------
ml_methods = ["Small CNN", "ResNet-50 (linear probe)", "ResNet-50 (fine-tuned)"]
main_name = max(ml_methods, key=lambda m: results[m]["dev"]["top1"])
main_model, main_size = results[main_name]["_model"], results[main_name]["_size"]
print("error analysis on:", main_name)
dev_probs = predict(main_model, X_dev, y_dev, main_size)
dev_pred = dev_probs.argmax(1)
wrong = np.where(dev_pred != y_dev)[0]
rng = np.random.default_rng(SEED)
sample = rng.choice(wrong, size=min(24, len(wrong)), replace=False) if len(wrong) else []

err_rows = []
for i in sample:
    top5 = np.argsort(-dev_probs[i])[:5]
    err_rows.append({"file": os.path.basename(dev_df.path[i]), "true": CLASSES[y_dev[i]],
                     "pred": CLASSES[dev_pred[i]], "conf": round(float(dev_probs[i, dev_pred[i]]), 3),
                     "true_in_top5": bool(y_dev[i] in top5),
                     "rf_correct": bool(results["RF (color + LBP)"]["_model"].predict(F_dev[i:i+1])[0] == y_dev[i]),
                     "notes": ""})
err_df = pd.DataFrame(err_rows)
err_df.to_csv(f"{OUT_DIR}/error_sample.csv", index=False)

shown = sample[:16]   # the figure shows 16 of the sampled errors; the CSV has all of them
cols = 8; rows = max(1, math.ceil(len(shown) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(6.75, 1.05 * rows))
for ax in np.atleast_1d(axes).ravel():
    ax.axis("off")
for ax, i in zip(np.atleast_1d(axes).ravel(), shown):
    ax.imshow(X_dev[i]); ax.set_title(f"T: {CLASSES[y_dev[i]]}\nP: {CLASSES[dev_pred[i]]}", fontsize=5, pad=2)
fig.tight_layout(pad=0.3); fig.savefig(f"{OUT_DIR}/fig_errors.pdf", dpi=200); plt.show()

cm = confusion_matrix(y_dev, dev_pred, labels=np.arange(C))
np.fill_diagonal(cm, 0)
pairs = sorted(((cm[a, b] + cm[b, a], a, b) for a in range(C) for b in range(a + 1, C)), reverse=True)
pairs = [p for p in pairs if p[0] > 0][:8]
conf_df = pd.DataFrame([{"class A": CLASSES[a], "class B": CLASSES[b], "confusions": n} for n, a, b in pairs])
conf_df.to_csv(f"{OUT_DIR}/confused_pairs.csv", index=False)
print(f"dev errors: {len(wrong)} of {len(y_dev)}  |  in sample, true class in top 5: "
      f"{err_df.true_in_top5.mean() if len(err_df) else float('nan'):.0%}")
print(conf_df.to_string(index=False))

In [ ]:
# ---------- Export tables and numbers for the LaTeX report ----------
def pct(v):
    return "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{100 * v:.1f}"

def tex_escape(s):
    return str(s).replace("&", r"\&").replace("_", r"\_").replace("%", r"\%")

order = ["Majority class", "LogReg (color)", "RF (color + LBP)", "Small CNN", "ResNet-50 (linear probe)", "ResNet-50 (fine-tuned)"]
best_dev = max(results[m]["dev"]["top1"] for m in order)
lines = [r"\begin{tabular}{lccc|ccc}", r"\toprule",
         r" & \multicolumn{3}{c|}{Dev} & \multicolumn{3}{c}{Test} \\",
         r"Method & Top-1 & Top-5 & F1 & Top-1 & Top-5 & F1 \\", r"\midrule"]
for m in order:
    d, t = results[m]["dev"], results[m].get("test", {})
    cells = [pct(d[k]) for k in ("top1", "top5", "macro_f1")] + [pct(t.get(k)) for k in ("top1", "top5", "macro_f1")]
    if d["top1"] == best_dev:
        cells[0] = r"\textbf{" + cells[0] + "}"
    lines.append(tex_escape(m) + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
open(f"{OUT_DIR}/table_main.tex", "w").write("\n".join(lines) + "\n")

lines = [r"\begin{tabular}{llcc}", r"\toprule", r"Method & Setting & Top-1 & F1 \\", r"\midrule"]
prev = None
for m, setting, a, f in sweeps:
    lines.append((tex_escape(m) if m != prev else "") + " & " + tex_escape(setting) + f" & {pct(a)} & {pct(f)} " + r"\\")
    prev = m
lines += [r"\bottomrule", r"\end{tabular}"]
open(f"{OUT_DIR}/table_sweep.tex", "w").write("\n".join(lines) + "\n")

lines = [r"\begin{tabular}{llc}", r"\toprule", r"Class A & Class B & Errors \\", r"\midrule"]
lines += [f"{tex_escape(r['class A'])} & {tex_escape(r['class B'])} & {r['confusions']} " + r"\\" for _, r in conf_df.head(6).iterrows()]
lines += [r"\bottomrule", r"\end{tabular}"]
open(f"{OUT_DIR}/table_confused.tex", "w").write("\n".join(lines) + "\n")

g = lambda m, s, k: pct(results[m].get(s, {}).get(k))
desc = counts.describe()
macros = {
    "NClasses": C, "NTrainFull": len(full_train), "NTrain": len(train_df), "NDev": len(dev_df), "NTest": len(test_df),
    "TrainMean": f'{desc.loc["mean", "train"]:.0f}', "TrainMin": int(desc.loc["min", "train"]), "TrainMax": int(desc.loc["max", "train"]),
    "TrainMed": int(desc.loc["50%", "train"]), "TestMin": int(desc.loc["min", "test"]), "TestMax": int(desc.loc["max", "test"]),
    "DevErrors": len(wrong), "ErrSampleN": len(err_df), "ErrShownN": min(16, len(err_df)),
    "ErrTopFive": pct(err_df.true_in_top5.mean()) if len(err_df) else "--",
    "MajDev": g("Majority class", "dev", "top1"),
    "LRDev": g("LogReg (color)", "dev", "top1"), "LRTest": g("LogReg (color)", "test", "top1"),
    "LRC": f"{results['LogReg (color)']['hparams']['C']:g}",
    "RFDev": g("RF (color + LBP)", "dev", "top1"), "RFTest": g("RF (color + LBP)", "test", "top1"),
    "RFTopFiveTest": g("RF (color + LBP)", "test", "top5"),
    "RFTrees": results["RF (color + LBP)"]["hparams"]["trees"], "RFLeaf": results["RF (color + LBP)"]["hparams"]["min_leaf"],
    "CNNDev": g("Small CNN", "dev", "top1"), "CNNTest": g("Small CNN", "test", "top1"),
    "CNNTopFiveTest": g("Small CNN", "test", "top5"), "CNNFOneTest": g("Small CNN", "test", "macro_f1"),
    "CNNLR": f"{cnn_hp['lr']:g}", "CNNJitter": "on" if cnn_hp["color_jitter"] else "off",
    "CNNEpochs": cnn_hp["epochs"], "CNNBestEpoch": cnn_hp["best_epoch"],
    "CNNTrainAcc": pct(float(cnn_hist.train_acc.iloc[-1])),
    "CNNParams": f"{sum(p.numel() for p in cnn.parameters())/1e6:.2f}",
    "CNNGap": pct(gap(cnn_hist)), "FTGap": pct(gap(ft_hist)),
    "LinDev": g("ResNet-50 (linear probe)", "dev", "top1"), "LinTest": g("ResNet-50 (linear probe)", "test", "top1"),
    "FTDev": g("ResNet-50 (fine-tuned)", "dev", "top1"), "FTTest": g("ResNet-50 (fine-tuned)", "test", "top1"),
    "FTTopFiveTest": g("ResNet-50 (fine-tuned)", "test", "top5"), "FTFOneTest": g("ResNet-50 (fine-tuned)", "test", "macro_f1"),
    "FTBackboneLR": f"{ft_hp['backbone_lr']:g}", "FTEpochs": ft_hp["epochs"], "FTBestEpoch": ft_hp["best_epoch"],
    "LinEpochs": TL_EPOCHS_LIN,
    "ErrModel": {"Small CNN": "the scratch CNN", "ResNet-50 (linear probe)": "the ResNet-50 linear probe",
                 "ResNet-50 (fine-tuned)": "the fine-tuned ResNet-50"}[main_name],
}
with open(f"{OUT_DIR}/macros.tex", "w") as fh:
    fh.write("% Generated by gemstone_midterm.ipynb. Do not edit by hand.\n")
    for k, v in macros.items():
        fh.write(f"\\newcommand{{\\{k}}}{{{v}}}\n")
    if QUICK:
        fh.write("\\newcommand{\\QuickRunWarning}{}\n")

import importlib.metadata as md
with open(f"{OUT_DIR}/environment.txt", "w") as fh:
    fh.write(f"python=={sys.version.split()[0]}\n")
    for pkg in ["torch", "torchvision", "numpy", "pandas", "scikit-learn", "scikit-image", "pillow", "matplotlib", "kagglehub"]:
        try:
            fh.write(f"{pkg}=={md.version(pkg)}\n")
        except md.PackageNotFoundError:
            pass
    fh.write(f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}\n")

json.dump({m: {k: v for k, v in r.items() if not k.startswith("_")} for m, r in results.items()},
          open(f"{OUT_DIR}/results.json", "w"), indent=2)
print(open(f"{OUT_DIR}/table_main.tex").read())
print(sorted(os.listdir(OUT_DIR)))

In [ ]:
# ---------- Download results.zip (Colab) ----------
shutil.make_archive("results", "zip", ".", OUT_DIR)
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Not in Colab: results.zip is in", os.getcwd())

# Optional: keep the model weights. They are too large for GitHub, so copy them to Google Drive.
SAVE_WEIGHTS_TO_DRIVE = False
if SAVE_WEIGHTS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/gemstone-recognition/runs"
    os.makedirs(dest, exist_ok=True)
    for f in os.listdir(RUNS_DIR):
        shutil.copy(os.path.join(RUNS_DIR, f), dest)
    print("weights copied to", dest)